In [1]:
import app
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
import rasterio
import rasterio.plot as rplt
import rasterio.windows  as rw
from rasterio.transform import Affine
from matplotlib_scalebar.scalebar import ScaleBar
from rasterio.warp import reproject, Resampling, calculate_default_transform
import numpy as np
import pandas as pd
import re
import cmocean as cmo

app.setup_logger(use_console_handler=True, use_file_handler=False)

In [2]:
base_directory = r"D:\PhD\21_Experiments\TidesDamageDriver"
drainages_folder = os.path.join(
    base_directory, "02_processed", "02_drainages"
)
data_img_folder = os.path.join(
    base_directory, "01_raw", "L8S2S1-events"
)
new_folder = os.path.join(
    base_directory, "02_processed", "03_drainages"
)

In [3]:
gdf_e = app.lakes.plotting.get_geodataframes(drainages_folder, ["_e."])[0]

In [4]:
check = [
    [0, "drainage", "2017-01-11", "2017-01-25", "L8", "L8", ""],
    [1, "clouds", "", "", "", "", ""],
    [2, "drainage", "2017-01-31", "2017-02-03", "S2", "L8", ""],
    [3, "drainage", "2017-01-11", "2017-01-18", "L8", "L8", ""],
    [4, "same", "2017-01-11", "2017-01-18", "L8", "L8", "same as 3"],
    [5, "drainage", "2017-01-16", "2017-01-23", "L8", "L8", ""],
    [6, "same", "2017-01-16", "2017-01-23", "L8", "L8", "same as 5"],
    [7, "runoff", "2017-01-11", "2017-01-25", "L8", "L8", "river like"],
    [8, "drainage", "2017-01-27", "2017-02-03", "L8", "L8", ""],
    [9, "freezing", "2017-01-11", "2017-01-25", "L8", "L8", ""],
    [10, "freezing", "2017-01-11", "2017-01-25", "L8", "L8", ""],
    [11, "freezing", "2017-01-10", "2017-01-18", "L8", "L8", ""],
    [12, "runoff", "", "", "", "", ""],
    [13, "runoff", "", "", "", "", ""],
    [14, "drainage", "2017-02-03", "2017-02-10", "L8", "L8", ""],
    [15, "drainage", "2017-02-03", "2017-02-10", "L8", "L8", ""],
    [16, "same", "2017-02-03", "2017-02-10", "L8", "L8", "same as 15"],
    [17, "drainage", "2019-01-31", "2019-02-08", "L8", "S2", ""],
    [18, "runoff", "", "", "", "", ""],
    [19, "drainage", "2020-01-18", "2020-01-21", "L8", "S1", ""],
    [20, "drainage", "2019-12-10", "2019-12-15", "L8", "S2", ""],
    [21, "drainage", "2019-12-19", "2019-12-22", "L8", "S2", ""],
    [22, "clouds", "", "", "", "", ""],
    [23, "drainage", "2020-01-04", "2020-01-11", "L8", "L8", ""],
    [24, "drainage", "2020-01-18", "2020-01-21", "L8", "S2", ""],
    [25, "runoff", "", "", "", "", ""],
    [26, "drainage", "2020-01-20", "2020-01-27", "L8", "L8", ""],
    [27, "drainage", "2020-01-18", "2020-01-24", "L8", "S2", ""],
    [28, "drainage", "2019-12-22", "2019-12-31", "L8", "S2", ""],
    [29, "drainage", "2019-12-24", "2019-12-31", "L8", "S2", ""],
    [30, "drainage", "2019-12-10", "2019-12-15", "L8", "S2", ""],
    [31, "clouds", "", "", "", "", ""],
    [32, "drainage", "2020-01-30", "2020-02-03", "S2", "S2", ""],
    [33, "freezing", "2021-01-31", "2021-02-07", "L8", "L8", ""],
    [34, "freezing", "2021-01-31", "2021-02-07", "L8", "L8", ""],
    [35, "drainage", "2021-02-07", "2021-02-09", "L8", "S1", ""],
    [36, "clouds", "", "", "", "", ""],
    [37, "drainage", "2021-01-05", "2021-01-15", "L8", "S2", ""],
    [38, "same", "2021-02-07", "2021-02-14", "L8", "L8", "same as 37"],
    [39, "drainage", "2020-01-07", "2020-01-13", "L8", "S2", ""],
    [40, "drainage", "2020-01-30", "2020-02-03", "S2", "S2", ""],
    [41, "drainage", "2020-01-04", "2020-01-11", "S2", "L8", ""],
    [42, "drainage", "2020-01-13", "2020-01-18", "L8", "S", "clouds"],
]

In [5]:
gdf_e["justification"] = [j[1] for j in check]
gdf_e["start_date"] = [j[2] for j in check]
gdf_e["end_date"] = [j[3] for j in check]
gdf_ee = gdf_e[gdf_e["justification"].isin(["drainage", "maybe"])].copy()
gdf_ee.reset_index(drop=False, inplace=True)

In [6]:
gdf_ee

,index,criteria,window,lake id,type,tile,ifile_0,area,file_0,date-0,...,volume-0,median-1,mean-1,volume-1,std_depth,year,geometry,justification,start_date,end_date
0,0,181_5_L8_2_L8_727_2017-01-14_2017-01-22_drain,0,727.0,drain,181,5,53100.0,tile-181_L8_2017-01-10_2017-01-18_30m.tif,2017-01-14,...,40620.850682,0.000000,0.000000,0.000000,NaN,2016,"MULTIPOLYGON (((2597640 -514590, 2597640 -5146...",drainage,2017-01-11,2017-01-25
1,2,181_6_S2_2_L8_584_2017-01-15_2017-01-22_drain,0,584.0,drain,181,6,60300.0,tile-181_S2_2017-01-10_2017-01-20_10m.tif,2017-01-15,...,39906.182206,0.000000,0.000000,0.000000,NaN,2016,"POLYGON ((2597550 -514080, 2597580 -514080, 25...",drainage,2017-01-31,2017-02-03
2,3,181_8_L8_1_S2_161_2017-01-30_2017-02-04_shrink,0,161.0,shrink,181,8,96300.0,tile-181_L8_2017-01-26_2017-02-03_30m.tif,2017-01-30,...,123562.726021,0.170166,0.172314,2016.074359,0.362372,2016,"POLYGON ((2576310 -471300, 2576340 -471300, 25...",drainage,2017-01-11,2017-01-18
3,5,182_5_L8_2_L8_575_2017-01-14_2017-01-22_shrink,0,575.0,shrink,182,5,36000.0,tile-182_L8_2017-01-10_2017-01-18_30m.tif,2017-01-14,...,271096.532965,0.860840,0.868432,37516.248894,0.535359,2016,"MULTIPOLYGON (((2556240 -445050, 2556240 -4451...",drainage,2017-01-16,2017-01-23
4,8,183_7_S2_2_L8_90_2017-01-15_2017-01-22_shrink,0,90.0,shrink,183,7,90900.0,tile-183_S2_2017-01-10_2017-01-20_10m.tif,2017-01-15,...,130987.440419,0.222290,0.222290,600.183105,0.717127,2016,"POLYGON ((2565840 -271140, 2565870 -271140, 25...",drainage,2017-01-27,2017-02-03
5,14,181_6_S2_2_L8_22_2017-01-15_2017-01-22_shrink,0,22.0,shrink,181,6,1800.0,tile-181_S2_2017-01-10_2017-01-20_10m.tif,2017-01-15,...,119482.474029,0.274536,0.291931,2890.118408,0.294078,2016,"MULTIPOLYGON (((2569200 -467880, 2569200 -4679...",drainage,2017-02-03,2017-02-10
6,15,181_8_L8_1_S2_388_2017-01-30_2017-02-04_shrink,0,388.0,shrink,181,8,900.0,tile-181_L8_2017-01-26_2017-02-03_30m.tif,2017-01-30,...,43381.439209,0.471680,0.500662,3604.769182,0.199416,2016,"MULTIPOLYGON (((2599470 -492540, 2599440 -4925...",drainage,2017-02-03,2017-02-10
7,17,182_13_L8_1_L8_300_2019-01-30_2019-02-07_shrink,0,300.0,shrink,182,13,99900.0,tile-182_L8_2019-01-26_2019-02-03_30m.tif,2019-01-30,...,91389.173824,0.585693,0.595006,7497.070205,0.263434,2018,"POLYGON ((2573160 -400260, 2573250 -400260, 25...",drainage,2019-01-31,2019-02-08
8,19,181_11_L8_1_L8_447_2020-01-22_2020-01-30_drain,0,447.0,drain,181,11,75600.0,tile-181_L8_2020-01-18_2020-01-26_30m.tif,2020-01-22,...,151999.203873,0.000000,0.000000,0.000000,NaN,2019,"POLYGON ((2583750 -477600, 2583780 -477600, 25...",drainage,2020-01-18,2020-01-21
9,20,181_2_L8_1_S2_53_2019-12-13_2019-12-16_shrink,0,53.0,shrink,181,2,118800.0,tile-181_L8_2019-12-09_2019-12-17_30m.tif,2019-12-13,...,139426.153421,0.479370,0.487222,7892.992795,0.339331,2019,"POLYGON ((2567100 -471300, 2567190 -471300, 25...",drainage,2019-12-10,2019-12-15


In [7]:
gdf_ee.to_file(os.path.join(new_folder, "drainages.shp"))

C:\Users\jsommer1\AppData\Local\Temp\ipykernel_28872\1840372279.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ee.to_file(os.path.join(new_folder, "drainages.shp"))
c:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\.venvDrainges\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field date-0 create as date field, though DateTime requested.
  ogr_write(
c:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\.venvDrainges\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field start-0 create as date field, though DateTime requested.
  ogr_write(
c:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\.venvDrainges\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field end-0 create as date field, though DateTime requested.
  ogr_write(
c:\Users\jsommer1\[Code]\HydrofractureShackleton_2023\.venvDrainges\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field date-1 create as date field, though DateTime reques